# Text experiments (GloVe) — Deep Multimodal K-fold Model

**Runtime → Change runtime type → GPU**, then **Runtime → Run all**. When the upload dialog appears, choose `kfold-multimodal-emotion-sentiment.zip`. Total time on a T4: ~15–20 min (GloVe download is most of it). At the end `text_results.zip` downloads automatically.

In [ ]:
# 1. Get the repository: upload the zip (or set REPO_URL to a GitHub URL once it is pushed)
import os, shutil, glob
REPO_URL = ""          # e.g. "https://github.com/<user>/kfold-multimodal-emotion-sentiment.git"
if REPO_URL:
    !git clone -q $REPO_URL repo
else:
    from google.colab import files
    up = files.upload()
    zipname = [k for k in up if k.endswith(".zip")][0]
    !rm -rf repo _unz && mkdir _unz && unzip -q "$zipname" -d _unz
    inner = [d for d in glob.glob("_unz/*") if os.path.isdir(d)][0]
    shutil.move(inner, "repo")
%cd repo
!ls

In [ ]:
# 2. Dependencies
!pip -q install imbalanced-learn nltk tabulate
import nltk
for r in ("punkt","punkt_tab","stopwords","wordnet","omw-1.4"):
    nltk.download(r, quiet=True)
import tensorflow as tf; print("TF", tf.__version__, "GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# 3. GloVe 840B-300d (~2 GB zip). Primary: Stanford; fallback: Hugging Face mirror.
import os
if not os.path.exists("data/glove.840B.300d.txt"):
    !wget -q --show-progress -O data/glove.zip https://nlp.stanford.edu/data/glove.840B.300d.zip \
      || wget -q --show-progress -O data/glove.zip https://huggingface.co/stanfordnlp/glove/resolve/main/glove.840B.300d.zip
    !cd data && unzip -q glove.zip && rm glove.zip
!ls -la data/glove.840B.300d.txt

In [ ]:
# 4. The four text experiments (Tables 4–5 and 8–9 of the paper). Each writes results/text_<task>_<model>/
import subprocess, itertools
for task, model in itertools.product(["emotion", "sentiment"], ["cnn", "lstm"]):
    print(f"\n===== text {task} {model} =====")
    !python run_experiment.py --modality text --task $task --model $model --glove data/glove.840B.300d.txt --verbose 0

In [ ]:
# 5. Summary table + download the results
!python -c "import sys; sys.path.insert(0,'src'); from kfold_mm.train import collect_results; print(collect_results('results').to_string(index=False))"
!rm -f results/*/model.keras
!zip -q -r ../text_results.zip results/text_emotion_cnn results/text_emotion_lstm results/text_sentiment_cnn results/text_sentiment_lstm results/results_table.md results/results_table.csv
from google.colab import files
files.download("../text_results.zip")